In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats
from iminuit import Minuit, cost

from matplotlib.markers import MarkerStyle

my_cmap = plt.colormaps["rainbow"]
my_marker=MarkerStyle(marker="1", fillstyle='full', transform=None, capstyle=None, joinstyle=None)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['savefig.dpi']=600
plt.rcParams["savefig.bbox"]='tight'
plt.rcParams["figure.figsize"]='5,3'
text_style = dict(horizontalalignment='right', verticalalignment='center',
                  fontsize=12, fontfamily='monospace')

r = np.random
r.seed(42)

## #7_1 Likelyhood and p-values


In [ ]:
def null_fct(x,rho,omega) :
    y=1+rho*x+omega*x**2
    return y

def alt_fct(x,rho,omega,gamma) :
    y=null_fct(x,rho,omega)-gamma*x**5
    return y




def norm_null_fct(x,rho,omega) :
    integral= (2 * (1 + omega / 3))
    y=null_fct(x,rho,omega)/integral
    return y

def norm_alt_fct(x,rho,omega,gamma) :
    integral= (2 * (1 + omega / 3))
    y=alt_fct(x,rho,omega,gamma)/integral
    return y




def neg_LLH_null(array,rho,omega):
    llh=0
    rej_count=0
    for x in array:
        if norm_null_fct(x,rho,omega) > 0:
            llh=-np.log(norm_null_fct(x,rho,omega))+llh
        else:
            rej_count+=1
    print("Null model: Number of rejected events:",rej_count)
    return llh

def neg_LLH_alt(array,rho,omega,gamma):
    llh=0
    rej_count=0
    for x in array:
        if norm_alt_fct(x,rho,omega,gamma) > 0:
            llh=-np.log(norm_alt_fct(x,rho,omega,gamma))+llh
        else:
            rej_count+=1
    print("Alternative model: Number of rejected events:",rej_count)
    return llh



def test_statistic(array,rho,omega,gamma) :
    llh_alt=neg_LLH_alt(array,rho,omega,gamma)
    llh_null=neg_LLH_null(array,rho,omega)
    return -2*(llh_null-llh_alt)


In [29]:
"""read in files"""

filename="LLH_Ratio_2_data.txt"
data=pd.read_csv(filename,sep=" ",header=None)
data=data.to_numpy()
x1=data[:,0]
x2=data[:,1]


## first column

In [ ]:
"""fitting parameters"""
rho_init=0.5
omega_init=0.2
gamma_init=0.1

cost_null=lambda rho,omega: neg_LLH_null(x1, rho, omega)
minuit_null=Minuit(cost_null,rho=rho_init,omega=omega_init)
minuit_null.migrad()


cost_alt=lambda rho,omega, gamma: neg_LLH_alt(x1, rho, omega, gamma)
minuit_alt=Minuit(cost_alt,rho=rho_init,omega=omega_init,gamma=gamma_init)
minuit_alt.migrad();

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.343e+04                  │              Nfcn = 75               │
│ EDM = 4.73e-05 (Goal: 0.0002)    │            time = 9.8 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ rho   │   0.32    │   0.04    │            │            │         │         │       │
│ 1 │ omega │   0.66    │   0.05    │            │            │         │         │       │
│ 2 │ gamma │   0.06    │   0.08    │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────┐
│       │     rho   omega   gamma │
├───────┼─────────────────────────┤
│   rho │ 0.00137  0.0002  0.0022 │
│ omega │  0.0002 0.00262 -0.0003 │
│ gamma │  0.0022 -0.0003 0.00562 │
└───────┴─────────────────────────┘

In [ ]:
print(f"-LLH null: {minuit_null.fval}")
print(f"-LLH alt: {minuit_alt.fval}")
test_stat=2*(minuit_null.fval-minuit_alt.fval)
print(f"Test statistic -2*(LLH_h0-LLH_hA): {test_stat}")
print(f"p-value: {stats.chi2.sf(test_stat, df=1)}")

#in agreement with the results from lecture

-LLH null: 13432.139550475775
-LLH alt: 13431.405461954653
Test statistic -2*(LLH_h0-LLH_hA): 1.4681770422430418
p-value: 0.22563353574263673


## second file

In [ ]:

filename="LLH_Ratio_2a_data.txt"
data=pd.read_csv(filename,sep=" ",header=None)
data=data.to_numpy()
x1=data[:,0]


rho_init=0.5
omega_init=0.2
gamma_init=0.1

cost_null=lambda rho,omega: neg_LLH_null(x1, rho, omega)
minuit_null=Minuit(cost_null,rho=rho_init,omega=omega_init)
minuit_null.migrad()


cost_alt=lambda rho,omega, gamma: neg_LLH_alt(x1, rho, omega, gamma)
minuit_alt=Minuit(cost_alt,rho=rho_init,omega=omega_init,gamma=gamma_init)
minuit_alt.migrad();

print(f"-LLH null: {minuit_null.fval}")
print(f"-LLH alt: {minuit_alt.fval}")
test_stat=2*(minuit_null.fval-minuit_alt.fval)
print(f"Test statistic -2*(LLH_h0-LLH_hA): {test_stat}")
print(f"p-value: {stats.chi2.sf(test_stat, df=1)}")

#in agreement with the results from lecture

-LLH null: 13651.005587956723
-LLH alt: 13495.017490163102
Test statistic -2*(LLH_h0-LLH_hA): 311.9761955872418
p-value: 8.104535434091065e-70
